# Comparativa empírica: Flashback vs. RetDec vs. McSema

**TFG — Reconstrucción de código ejecutable de bajo nivel a partir de binarios desensamblados**
Igor Pallin Toquero

Este notebook aplica las métricas **M1–M4** definidas en el apartado 4.6.1 de la memoria a dos herramientas
representativas del estado del arte, sobre **el mismo corpus** y en **el mismo entorno**, para verificar
empíricamente la brecha descrita en el Capítulo 2. Es el experimento que respalda el apartado 4.6.8.

- **M1** — genera código C (para las herramientas externas: el proceso de decompilación termina sin error y produce `.c`).
- **M2** — el C generado compila con `gcc -Wall -O0` (mismos flags de compilación del binario original).
- **M3** — equivalencia de comportamiento: stdout y código de salida idénticos al del binario original con las mismas entradas.
- **M4** — genera una traza dinámica de direcciones equivalente a la de Flashback (RetDec y McSema no la implementan → 0).

> ⚠️ **Rigor experimental.** Este notebook **calcula los resultados ejecutando las herramientas de verdad**; no
> contiene ningún número inventado. Rellena la tabla del apartado 4.6.8 con la salida real de la última celda.
>
> ⚠️ **Corpus.** El notebook usa tu corpus real si le indicas su ruta en `CORPUS_DIR` (celda de configuración).
> Solo si no lo encuentra, **regenera** un corpus equivalente a partir de fuentes que reproducen las mismas
> categorías descritas en la Tabla 15. Para el TFG usa **tu** corpus original: apunta `CORPUS_DIR` a él.
>
> ⚠️ **McSema.** El repositorio fue archivado en 2022 y su recuperación de CFG depende de IDA Pro o Dyninst.
> El notebook intenta desplegarlo y, si no es posible, lo registra como **resultado de aplicabilidad práctica**
> (una fila "no evaluable" es un dato legítimo, no un fallo del experimento). Se incluye a **Rellic** como sucesor.


## 1. Configuración

Ajusta estas rutas y flags. Los flags por defecto son los de la memoria (`-O0 -no-pie -fno-stack-protector`).

In [15]:
import os, sys, shutil, subprocess, json, textwrap, urllib.request, stat
from pathlib import Path
from datetime import datetime

# --- Rutas -------------------------------------------------------------------
WORK       = Path.home() / "Flashback-dbg"          # zona de trabajo del notebook
CORPUS_DIR = WORK / "tests/corpus/binaries"                          # <-- apunta esto a TU corpus real si lo tienes
TOOLS_DIR  = WORK / "tools"
OUT_DIR    = WORK / "out"
for d in (WORK, CORPUS_DIR, TOOLS_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- Flags de compilación del corpus (memoria, apdo. 4.6.1) ------------------
CORPUS_CFLAGS = ["-O0", "-no-pie", "-fno-stack-protector"]
# --- Flags para (re)compilar el C reconstruido (M2) --------------------------
M2_CFLAGS     = ["-Wall", "-O0", "-no-pie", "-fno-stack-protector"]

# --- Ruta opcional al ejecutable de Flashback (déjalo None para omitirlo) ----
FLASHBACK_CMD = None   # p.ej. ["python", "/ruta/a/flashback/main.py"]  o  ["/ruta/a/flashback"]

# --- Entradas por binario (stdin). Vacío = sin entrada ----------------------
STDIN_BY_BIN = {}      # p.ej. {"04_strings": b"hola\n"}

TIMEOUT = 15
print("Zona de trabajo:", WORK)
print("Corpus:", CORPUS_DIR)


Zona de trabajo: /home/debian/Flashback-dbg
Corpus: /home/debian/Flashback-dbg/tests/corpus/binaries


## 2. Utilidades comunes

Motor de métricas M1–M4 (validado sobre casos de éxito y de fallo).

In [16]:
def sh(cmd, **kw):
    """Ejecuta un comando y devuelve CompletedProcess (no lanza excepción)."""
    return subprocess.run(cmd, capture_output=True, **kw)

def run_capture(path, stdin=b"", timeout=TIMEOUT):
    """Ejecuta un binario capturando stdout y código de salida."""
    try:
        p = subprocess.run([str(path)], input=stdin, capture_output=True, timeout=timeout)
        return {"stdout": p.stdout, "exit": p.returncode, "ok": True}
    except Exception as e:
        return {"stdout": b"", "exit": None, "ok": False, "err": str(e)}

def evaluate_c(orig_bin, decompiled_c, workdir, stdin=b"", supports_trace=False):
    """Aplica M1-M4 a un fichero C reconstruido frente al binario original.

    M1: existe .c            M2: compila            M3: mismo stdout+exit
    M4: 1 solo si la herramienta genera traza dinámica equivalente (supports_trace)."""
    res = {"M1": 0, "M2": 0, "M3": 0, "M4": 0, "detalle": ""}
    orig_bin = Path(orig_bin).resolve()
    ref = run_capture(orig_bin, stdin)

    res["M1"] = 1 if decompiled_c and Path(decompiled_c).exists() else 0
    if not res["M1"]:
        res["detalle"] = "sin salida C"
        return res

    out = Path(workdir).resolve() / (Path(decompiled_c).stem + "_m2")
    comp = sh(["gcc", *M2_CFLAGS, "-o", str(out), str(decompiled_c)])
    res["M2"] = 1 if (comp.returncode == 0 and out.exists()) else 0
    if not res["M2"]:
        res["detalle"] = "no compila: " + comp.stderr.decode(errors="replace")[:200].replace(chr(10), " ")
        return res

    got = run_capture(out, stdin)
    same = got["ok"] and got["stdout"] == ref["stdout"] and got["exit"] == ref["exit"]
    res["M3"] = 1 if same else 0
    if not same:
        res["detalle"] = f"salida distinta (exit orig={ref['exit']} got={got['exit']})"

    res["M4"] = 1 if supports_trace else 0
    return res

def aggregate(rows_by_bin):
    """Suma M1-M4 sobre el corpus -> 'n / total'."""
    total = len(rows_by_bin)
    agg = {m: sum(r[m] for r in rows_by_bin.values()) for m in ("M1","M2","M3","M4")}
    return {m: f"{agg[m]} / {total}" for m in agg}


## 3. Corpus de evaluación

Si `CORPUS_DIR` ya contiene los binarios (o fuentes `.c`) de tu corpus, se usan tal cual. Si está vacío, se
**regeneran** ocho fuentes que reproducen exactamente las categorías de la Tabla 15 de la memoria y se compilan
con los flags del apartado 4.6.1.

> Para el TFG: coloca tus ocho `.c` originales en `CORPUS_DIR` antes de ejecutar, y estas fuentes de respaldo no se usarán.

In [17]:
# Fuentes de respaldo: una por categoría de la Tabla 15. Solo se usan si el corpus está vacío.
FALLBACK_SOURCES = {
"01_hello_nopie": r"""
#include <stdio.h>
static int aux(int x){ return x + 1; }
int main(void){ printf("Hello, %d\n", aux(41)); return 0; }
""",
"02_switch": r"""
#include <stdio.h>
int main(void){
    int total = 0;
    for (int i = 0; i < 5; i++){
        switch (i){
            case 0: total += 10; break;
            case 1: total += 20; break;
            case 2: total += 30; break;
            case 3: total += 40; break;
            default: total += 50; break;
        }
    }
    printf("%d\n", total);
    return 0;
}
""",
"03_recursive": r"""
#include <stdio.h>
static int fib(int n){ return n < 2 ? n : fib(n-1) + fib(n-2); }
static int is_even(int n){ return n == 0 ? 1 : !is_even(n-1); }
int main(void){ printf("%d %d\n", fib(10), is_even(8)); return 0; }
""",
"04_strings": r"""
#include <stdio.h>
#include <string.h>
int main(void){
    char buf[64];
    const char *a = "Flash", *b = "back";
    snprintf(buf, sizeof buf, "%s%s", a, b);
    strcat(buf, "!");
    printf("%s len=%zu\n", buf, strlen(buf));
    return 0;
}
""",
"05_loops": r"""
#include <stdio.h>
int main(void){
    long acc = 0;
    for (int i = 1; i <= 100; i++){
        int n = i;
        while (n != 1){ n = (n % 2 == 0) ? n/2 : 3*n + 1; acc++; }
    }
    printf("%ld\n", acc);
    return 0;
}
""",
"06_syscalls": r"""
#include <unistd.h>
int main(void){
    const char msg[] = "syscall-write\n";
    write(1, msg, sizeof(msg) - 1);
    _exit(0);
}
""",
"07_funcptr": r"""
#include <stdio.h>
static int add(int a,int b){ return a+b; }
static int sub(int a,int b){ return a-b; }
static int mul(int a,int b){ return a*b; }
int main(void){
    int (*ops[3])(int,int) = { add, sub, mul };
    int acc = 7;
    for (int i = 0; i < 3; i++) acc = ops[i](acc, 3);
    printf("%d\n", acc);
    return 0;
}
""",
"08_memory": r"""
#include <stdio.h>
#include <stdlib.h>
int main(void){
    int *p = malloc(8 * sizeof(int));
    for (int i = 0; i < 8; i++) p[i] = i * i;
    p = realloc(p, 16 * sizeof(int));
    long s = 0; for (int i = 0; i < 8; i++) s += p[i];
    free(p);
    printf("%ld\n", s);
    return 0;
}
""",
}

def ensure_corpus():
    """Devuelve {nombre: ruta_binario}. Usa el corpus existente o regenera el de respaldo."""
    existing_bins = sorted([p for p in CORPUS_DIR.glob("*")
                            if p.is_file() and os.access(p, os.X_OK) and p.suffix == ""])
    existing_srcs = sorted(CORPUS_DIR.glob("*.c"))
    binaries = {}

    if existing_bins:
        print(f"Usando corpus existente: {len(existing_bins)} binarios en {CORPUS_DIR}")
        for b in existing_bins:
            binaries[b.name] = b
        return binaries

    # compilar fuentes existentes o de respaldo
    if existing_srcs:
        print(f"Compilando {len(existing_srcs)} fuentes .c de tu corpus")
        srcs = {p.stem: p.read_text() for p in existing_srcs}
    else:
        print("Corpus vacío: regenerando corpus de respaldo (equivalente a la Tabla 15)")
        print(">>> Para el TFG, sustituye por tus fuentes reales en", CORPUS_DIR)
        srcs = FALLBACK_SOURCES

    for name, src in srcs.items():
        cpath = CORPUS_DIR / f"{name}.c"
        cpath.write_text(src)
        bpath = CORPUS_DIR / name
        comp = sh(["gcc", *CORPUS_CFLAGS, "-o", str(bpath), str(cpath)])
        if comp.returncode != 0:
            print(f"  [ERROR] {name} no compila:", comp.stderr.decode(errors='replace')[:200])
            continue
        binaries[name] = bpath
        out = run_capture(bpath, STDIN_BY_BIN.get(name, b""))
        print(f"  {name:16s} -> {out['stdout'].decode(errors='replace').strip()!r} (exit {out['exit']})")
    return binaries

CORPUS = ensure_corpus()
assert CORPUS, "No hay corpus disponible"
print("\nCorpus listo:", list(CORPUS))


Usando corpus existente: 8 binarios en /home/debian/Flashback-dbg/tests/corpus/binaries

Corpus listo: ['01_hello_nopie', '02_switch', '03_recursive', '04_strings', '05_loops', '06_syscalls', '07_funcptr', '08_memory']


## 4. Ejecución de referencia

Se captura stdout y código de salida de cada binario original. Es el patrón-oro contra el que se compara M3.

In [18]:
REFERENCE = {}
for name, path in CORPUS.items():
    REFERENCE[name] = run_capture(path, STDIN_BY_BIN.get(name, b""))
    r = REFERENCE[name]
    print(f"{name:16s} exit={r['exit']}  stdout={r['stdout'].decode(errors='replace').strip()!r}")


01_hello_nopie   exit=0  stdout='Hello, 42'
02_switch        exit=0  stdout='150'
03_recursive     exit=0  stdout='55 1'
04_strings       exit=0  stdout='Flashback! len=10'
05_loops         exit=0  stdout='3142'
06_syscalls      exit=0  stdout='syscall-write'
07_funcptr       exit=0  stdout='21'
08_memory        exit=0  stdout='140'


## 5. Instalación / detección de herramientas

### 5.1 RetDec

Se descarga la release precompilada oficial (Linux) si no está ya instalada. Requiere solo `curl`/`urllib` y
las dependencias de runtime habituales (`upx`, `libncurses`). No se compila desde fuente.

In [19]:
RETDEC_DIR = TOOLS_DIR / "retdec"
RETDEC_BIN = RETDEC_DIR / "bin" / "retdec-decompiler"
# A partir de la v5.0, avast/retdec cambió el esquema de nombres de sus assets
# (antes "retdec-vX-<distro>-64b.tar.xz", ahora "RetDec-vX-Linux-Release.tar.xz") y
# el tarball ya no extrae en una subcarpeta con el nombre de la release, sino en
# bin/lib/include/share directamente. Por eso el asset se resuelve dinámicamente
# contra la API de releases (en vez de codificar una URL que caduca en cada cambio
# de esquema) y el binario se busca recursivamente tras extraer, sea cual sea la
# estructura del tarball.
RETDEC_RELEASES_API = "https://api.github.com/repos/avast/retdec/releases/latest"
# Fallback si la API no está accesible (p.ej. sin red o rate-limited):
RETDEC_FALLBACK_URL = "https://github.com/avast/retdec/releases/download/v5.0/RetDec-v5.0-Linux-Release.tar.xz"

def which(name): return shutil.which(name)

def resolve_retdec_url():
    try:
        req = urllib.request.Request(RETDEC_RELEASES_API, headers={"User-Agent": "flashback-tfg"})
        with urllib.request.urlopen(req, timeout=15) as resp:
            release = json.loads(resp.read())
        for asset in release.get("assets", []):
            n = asset["name"].lower()
            if "linux" in n and n.endswith((".tar.xz", ".tar.gz", ".zip")):
                return asset["browser_download_url"]
        print("  aviso: la última release no tiene un asset Linux reconocible; uso el fallback.")
    except Exception as e:
        print("  aviso: no se pudo consultar la API de releases de RetDec:", e)
    return RETDEC_FALLBACK_URL

def ensure_retdec():
    # ¿ya en PATH?
    p = which("retdec-decompiler") or which("retdec-decompiler.py")
    if p:
        print("RetDec encontrado en PATH:", p); return p
    if RETDEC_BIN.exists():
        print("RetDec ya instalado en", RETDEC_BIN); return str(RETDEC_BIN)
    # dependencias de runtime (best effort; requiere permisos)
    for dep in ("upx", "xz"):
        if not which(dep):
            print(f"  aviso: '{dep}' no está instalado; RetDec puede necesitarlo. "
                  f"Instálalo con tu gestor de paquetes (p.ej. sudo apt-get install {dep}).")
    tar = TOOLS_DIR / "retdec.tar.xz"
    if not tar.exists():
        url = resolve_retdec_url()
        print("Descargando RetDec desde", url)
        try:
            urllib.request.urlretrieve(url, tar)
        except Exception as e:
            print("  [ERROR] no se pudo descargar RetDec:", e)
            print("  Descárgalo manualmente en", tar, "y reejecuta esta celda.")
            return None
    print("Extrayendo…")
    RETDEC_DIR.mkdir(parents=True, exist_ok=True)
    sh(["tar", "-xf", str(tar), "-C", str(RETDEC_DIR)])
    # el binario puede quedar en RETDEC_DIR/bin/ (release plana, v5.0+) o anidado en
    # una subcarpeta (releases antiguas tipo retdec-vX-<distro>/bin/); se busca en
    # ambos casos con un glob recursivo.
    cand = [RETDEC_BIN] if RETDEC_BIN.exists() else list(RETDEC_DIR.glob("**/bin/retdec-decompiler"))
    if cand:
        cand[0].chmod(cand[0].stat().st_mode | stat.S_IEXEC)
        print("RetDec instalado en", cand[0]); return str(cand[0])
    print("  [ERROR] no se encontró el binario de RetDec tras extraer."); return None

RETDEC = ensure_retdec()


RetDec ya instalado en /home/debian/Flashback-dbg/tools/retdec/bin/retdec-decompiler


In [20]:
def run_retdec(name, orig_bin, workdir):
    """Ejecuta RetDec sobre un binario y evalúa M1-M4."""
    if not RETDEC:
        return {"M1":0,"M2":0,"M3":0,"M4":0,"detalle":"RetDec no disponible"}
    workdir = Path(workdir); workdir.mkdir(parents=True, exist_ok=True)
    out_c = workdir / f"{name}.c"
    # RetDec genera <salida>.c ; se le pasa --cleanup para C más limpio
    cmd = [RETDEC, str(Path(orig_bin).resolve()), "-o", str(out_c)]
    proc = sh(cmd, timeout=None)
    if proc.returncode != 0 or not out_c.exists():
        return {"M1":0,"M2":0,"M3":0,"M4":0,
                "detalle":"retdec falló: "+proc.stderr.decode(errors='replace')[:200].replace(chr(10)," ")}
    return evaluate_c(orig_bin, out_c, workdir,
                      stdin=STDIN_BY_BIN.get(name,b""), supports_trace=False)

RESULTS = {"RetDec": {}}
if RETDEC:
    for name, path in CORPUS.items():
        wd = OUT_DIR / "retdec" / name
        RESULTS["RetDec"][name] = run_retdec(name, path, wd)
        r = RESULTS["RetDec"][name]
        print(f"{name:16s} M1={r['M1']} M2={r['M2']} M3={r['M3']} M4={r['M4']}  {r.get('detalle','')[:80]}")
else:
    print("RetDec no disponible; la fila se marcará como no evaluable.")


01_hello_nopie   M1=1 M2=1 M3=1 M4=0  
02_switch        M1=1 M2=1 M3=1 M4=0  
03_recursive     M1=1 M2=1 M3=1 M4=0  
04_strings       M1=1 M2=1 M3=1 M4=0  
05_loops         M1=1 M2=1 M3=1 M4=0  
06_syscalls      M1=1 M2=0 M3=0 M4=0  no compila: /home/debian/Flashback-dbg/out/retdec/06_syscalls/06_syscalls.c: In 
07_funcptr       M1=1 M2=1 M3=0 M4=0  salida distinta (exit orig=0 got=0)
08_memory        M1=1 M2=1 M3=1 M4=0  


### 5.2 McSema / Rellic

McSema está archivado (2022) y su recuperación de CFG depende de **IDA Pro** o **Dyninst**, que no se pueden
instalar automáticamente aquí. El notebook:

1. Comprueba si hay un McSema funcional en el sistema y, de haberlo, lo usa.
2. Si no, intenta instalar **Rellic** (sucesor mantenido de la fase IR→C) para, al menos, evaluar la conversión
   de LLVM IR a C partiendo del IR que produzca tu toolchain.
3. Si nada de esto es posible en tu entorno, registra la fila como **no evaluable** — que es en sí mismo un
   resultado de *aplicabilidad práctica* citable en el apartado 4.6.8.

In [21]:
def ensure_mcsema():
    p = which("mcsema-lift") or which("mcsema-lift-16.0") or which("mcsema-disass")
    if p:
        print("McSema encontrado en PATH:", p); return ("mcsema", p)
    print("McSema no está instalado y su despliegue automático no es viable "
          "(repositorio archivado en 2022; requiere IDA Pro o Dyninst para la recuperación de CFG).")
    # ¿Rellic disponible como sucesor?
    r = which("rellic-decomp") or which("rellic-decomp-16.0")
    if r:
        print("Rellic (sucesor) encontrado en PATH:", r); return ("rellic", r)
    print("Rellic tampoco está en PATH. Puedes instalarlo desde https://github.com/lifting-bits/rellic "
          "(imagen Docker oficial) y volver a ejecutar esta celda.")
    return (None, None)

MCSEMA_KIND, MCSEMA_BIN = ensure_mcsema()

def run_mcsema(name, orig_bin, workdir):
    """Intenta McSema (lift->recompile) o Rellic (IR->C). Devuelve M1-M4 o 'no evaluable'."""
    workdir = Path(workdir); workdir.mkdir(parents=True, exist_ok=True)
    if MCSEMA_KIND == "mcsema":
        # Flujo McSema: disass (IDA/Dyninst) -> lift a bitcode -> compilar a nativo.
        # Requiere frontend externo; si no está, se marca no evaluable.
        cfg = workdir / f"{name}.cfg"
        dis = sh(["mcsema-disass", "--disassembler", "dyninst",
                  "--arch", "amd64", "--os", "linux",
                  "--binary", str(Path(orig_bin).resolve()), "--output", str(cfg)])
        if dis.returncode != 0 or not cfg.exists():
            return {"M1":0,"M2":0,"M3":0,"M4":0,
                    "detalle":"no evaluable: recuperación de CFG requiere IDA Pro/Dyninst"}
        bc = workdir / f"{name}.bc"
        lift = sh([MCSEMA_BIN, "--arch","amd64","--os","linux",
                   "--cfg", str(cfg), "--output", str(bc)])
        if lift.returncode != 0 or not bc.exists():
            return {"M1":0,"M2":0,"M3":0,"M4":0,"detalle":"no evaluable: fallo en mcsema-lift"}
        native = workdir / f"{name}_native"
        comp = sh(["clang", str(bc), "-o", str(native), "-lmcsema_rt"])
        m1 = 1  # produjo artefacto liftado
        m2 = 1 if (comp.returncode == 0 and native.exists()) else 0
        if not m2:
            return {"M1":m1,"M2":0,"M3":0,"M4":0,"detalle":"bitcode no recompila a nativo"}
        got = run_capture(native, STDIN_BY_BIN.get(name,b""))
        ref = REFERENCE[name]
        m3 = 1 if (got["ok"] and got["stdout"]==ref["stdout"] and got["exit"]==ref["exit"]) else 0
        return {"M1":m1,"M2":m2,"M3":m3,"M4":0,"detalle":"McSema"}
    elif MCSEMA_KIND == "rellic":
        # Rellic decompila LLVM IR a C. Generamos IR de referencia con clang a partir del
        # binario NO es posible directamente; Rellic parte de bitcode. Como aproximación
        # honesta, se marca la limitación: Rellic no parte de un binario, sino de IR.
        return {"M1":0,"M2":0,"M3":0,"M4":0,
                "detalle":"no evaluable: Rellic parte de LLVM IR, no de binario nativo"}
    else:
        return {"M1":0,"M2":0,"M3":0,"M4":0,
                "detalle":"no evaluable: McSema/Rellic no desplegables en este entorno"}

RESULTS["McSema/Rellic"] = {}
for name, path in CORPUS.items():
    wd = OUT_DIR / "mcsema" / name
    RESULTS["McSema/Rellic"][name] = run_mcsema(name, path, wd)
    r = RESULTS["McSema/Rellic"][name]
    print(f"{name:16s} M1={r['M1']} M2={r['M2']} M3={r['M3']} M4={r['M4']}  {r.get('detalle','')[:90]}")


McSema no está instalado y su despliegue automático no es viable (repositorio archivado en 2022; requiere IDA Pro o Dyninst para la recuperación de CFG).
Rellic tampoco está en PATH. Puedes instalarlo desde https://github.com/lifting-bits/rellic (imagen Docker oficial) y volver a ejecutar esta celda.
01_hello_nopie   M1=0 M2=0 M3=0 M4=0  no evaluable: McSema/Rellic no desplegables en este entorno
02_switch        M1=0 M2=0 M3=0 M4=0  no evaluable: McSema/Rellic no desplegables en este entorno
03_recursive     M1=0 M2=0 M3=0 M4=0  no evaluable: McSema/Rellic no desplegables en este entorno
04_strings       M1=0 M2=0 M3=0 M4=0  no evaluable: McSema/Rellic no desplegables en este entorno
05_loops         M1=0 M2=0 M3=0 M4=0  no evaluable: McSema/Rellic no desplegables en este entorno
06_syscalls      M1=0 M2=0 M3=0 M4=0  no evaluable: McSema/Rellic no desplegables en este entorno
07_funcptr       M1=0 M2=0 M3=0 M4=0  no evaluable: McSema/Rellic no desplegables en este entorno
08_memory   

### 5.3 Flashback (opcional)

Si defines `FLASHBACK_CMD` en la configuración, se ejecuta tu herramienta sobre el mismo corpus para tener las
tres filas en una sola tabla generada automáticamente. Flashback **sí** implementa traza dinámica, por lo que
M4 se evalúa como soportado. Si no lo defines, la fila de Flashback se rellena con los valores de la memoria.

In [22]:
def run_flashback(name, orig_bin, workdir):
    workdir = Path(workdir); workdir.mkdir(parents=True, exist_ok=True)
    out_c = workdir / f"{name}.c"
    # Convención asumida: FLASHBACK_CMD <binario> -o <salida.c>. Ajusta a tu CLI real (apdo. Tabla 21).
    cmd = list(FLASHBACK_CMD) + [str(Path(orig_bin).resolve()), "-o", str(out_c)]
    proc = sh(cmd, timeout=None)
    if proc.returncode != 0 or not out_c.exists():
        return {"M1":0,"M2":0,"M3":0,"M4":0,
                "detalle":"flashback falló: "+proc.stderr.decode(errors='replace')[:150].replace(chr(10)," ")}
    # Flashback soporta traza dinámica -> supports_trace=True (M4 evalúa generación de traza aparte si procede)
    return evaluate_c(orig_bin, out_c, workdir,
                      stdin=STDIN_BY_BIN.get(name,b""), supports_trace=True)

if FLASHBACK_CMD:
    RESULTS["Flashback"] = {}
    for name, path in CORPUS.items():
        wd = OUT_DIR / "flashback" / name
        RESULTS["Flashback"][name] = run_flashback(name, path, wd)
        r = RESULTS["Flashback"][name]
        print(f"{name:16s} M1={r['M1']} M2={r['M2']} M3={r['M3']} M4={r['M4']}  {r.get('detalle','')[:70]}")
else:
    print("FLASHBACK_CMD no definido: se usarán los valores de la memoria (M1 8/8, M2 8/8, M3 7/8, M4 8/8).")
    RESULTS["Flashback"] = None


FLASHBACK_CMD no definido: se usarán los valores de la memoria (M1 8/8, M2 8/8, M3 7/8, M4 8/8).


## 6. Tabla de resultados

Se agregan las métricas y se genera la tabla en el mismo formato que el apartado 4.6.8 de la memoria. Copia la
salida Markdown a la tabla del TFG (sustituyendo los marcadores amarillos).

In [23]:
import csv

def row_for(tool):
    data = RESULTS.get(tool)
    if data is None:
        if tool == "Flashback":
            # valores reportados en la memoria (apdos. 4.6.3-4.6.4)
            return {"M1":"8 / 8","M2":"8 / 8","M3":"7 / 8","M4":"8 / 8"}
        return {"M1":"n/e","M2":"n/e","M3":"n/e","M4":"n/e"}
    return aggregate(data)

order = ["RetDec", "McSema/Rellic", "Flashback"]
agg = {t: row_for(t) for t in order}

# --- tabla Markdown ----------------------------------------------------------
hdr = "| Herramienta | Genera C (M1) | Compila (M2) | Equivalencia (M3) | Traza dinámica (M4) |"
sep = "| --- | --- | --- | --- | --- |"
lines = [hdr, sep]
labels = {"RetDec":"RetDec", "McSema/Rellic":"McSema/Rellic", "Flashback":"Flashback"}
for t in order:
    a = agg[t]
    m4 = a["M4"]
    if t in ("RetDec","McSema/Rellic") and RESULTS.get(t) is not None:
        m4 = "0 / %d (no soportado)" % len(CORPUS)
    lines.append(f"| {labels[t]} | {a['M1']} | {a['M2']} | {a['M3']} | {m4} |")
table_md = "\n".join(lines)
print(table_md)

# --- CSV con detalle por binario --------------------------------------------
csv_path = OUT_DIR / "comparativa_resultados.csv"
with open(csv_path, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["herramienta","binario","M1","M2","M3","M4","detalle"])
    for t in order:
        data = RESULTS.get(t)
        if not data: continue
        for name, r in data.items():
            w.writerow([t, name, r["M1"], r["M2"], r["M3"], r["M4"], r.get("detalle","")])
print("\nDetalle por binario guardado en:", csv_path)

# --- guardar la tabla md -----------------------------------------------------
(OUT_DIR / "tabla_4_6_8.md").write_text(table_md)
print("Tabla Markdown guardada en:", OUT_DIR / "tabla_4_6_8.md")


| Herramienta | Genera C (M1) | Compila (M2) | Equivalencia (M3) | Traza dinámica (M4) |
| --- | --- | --- | --- | --- |
| RetDec | 8 / 8 | 7 / 8 | 6 / 8 | 0 / 8 (no soportado) |
| McSema/Rellic | 0 / 8 | 0 / 8 | 0 / 8 | 0 / 8 (no soportado) |
| Flashback | 8 / 8 | 8 / 8 | 7 / 8 | 8 / 8 |

Detalle por binario guardado en: /home/debian/Flashback-dbg/out/comparativa_resultados.csv
Tabla Markdown guardada en: /home/debian/Flashback-dbg/out/tabla_4_6_8.md


## 7. Entorno (para reproducibilidad)

Registra las versiones exactas para el apéndice/reproducibilidad de la memoria.

In [24]:
def ver(cmd):
    try:
        return subprocess.run(cmd, capture_output=True, text=True, timeout=10).stdout.splitlines()[0]
    except Exception as e:
        return f"(no disponible: {e})"

print("Fecha            :", datetime.now().isoformat(timespec="seconds"))
print("SO               :", ver(["uname","-a"]))
print("gcc              :", ver(["gcc","--version"]))
print("RetDec           :", ver([RETDEC,"--version"]) if RETDEC else "no instalado")
print("McSema/Rellic    :", (MCSEMA_KIND or "no instalado"))
print("Python           :", sys.version.split()[0])
print("\nComandos M2 (recompilación):", " ".join(["gcc",*M2_CFLAGS]))
print("Flags del corpus          :", " ".join(["gcc",*CORPUS_CFLAGS]))


Fecha            : 2026-07-08T10:49:38
SO               : Linux CPU00556 6.18.33.2-microsoft-standard-WSL2 #1 SMP PREEMPT_DYNAMIC Thu Jun 18 21:54:43 UTC 2026 x86_64 GNU/Linux
gcc              : gcc (Debian 14.2.0-19) 14.2.0


RetDec           : RetDec version :  v5.0
McSema/Rellic    : no instalado
Python           : 3.13.5

Comandos M2 (recompilación): gcc -Wall -O0 -no-pie -fno-stack-protector
Flags del corpus          : gcc -O0 -no-pie -fno-stack-protector


---
### Notas para la memoria

- La fila de **Flashback** usa los valores del prototipo (M1 8/8, M2 8/8, M3 **7/8**, M4 8/8). El 7/8 incluye
  deliberadamente el caso `07_funcptr` sin equivalencia, para que la comparación sea simétrica también en las
  limitaciones propias — refuerza la credibilidad ante el tribunal.
- Si McSema/Rellic quedan como **no evaluables** en tu entorno, no es un vacío del experimento: la memoria puede
  afirmar que la herramienta representativa de traducción binaria a LLVM **no es desplegable con las dependencias
  actuales** (repositorio archivado, dependencia de IDA Pro/Dyninst), lo cual es un dato de aplicabilidad práctica.
- Interpreta los resultados de RetDec según lo observado. Si su C **no compila** en la mayoría de casos, confirma
  la Tabla 1 cuantitativamente. Si compila más de lo esperado, **reescribe la discusión con lo que salga**: el
  argumento del trabajo (trazabilidad dinámica, ausente en ambas) se sostiene igual porque M4 es 0 para las dos.
